Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Exercise03

## Problem

In the last exercise, you used tabular analyses to examine how eviction impacts tenants in Montgomery and Prince George's Counties. In this exercise, you will use more precise, address-level data to analyze evictions across the entire state through a spatial lens.

**You get to write your own research question, but with a few constraints:**
- It should be addressable with the provided eviction data
- It should require that you relate the eviction data to at least one other dataset, which you provide
- It should involve at least one form of spatial analysis (e.g., proximity, overlay, or measurement of another spatial relationship)

**Please write a short abstract (200-300 words) at the top of your exercise notebook that concisely summarizes your research question, how you addressed it, and the results of your analysis. Then provide reproducible code in cells below.**

### Bonus

Only a portion of the eviction records I'm providing for this exercise have addresses that can be geocoded (converting address strings to geographic coordinates) with a high degree of accuracy, or even at all. How could you assess bias in which records are accurately geocoded? (Hint: This will require you to define accuracy.) Can you write a Python script that evaluates whether higher- and lower-accuracy geocodes are randomly distributed across eviction records, or whether certain types of evictions are more or less likely to be geocoded well? **Please report your approach and findings in a separate paragraph and provide supporting code.**

## Data

[Exercise 3 Google Drive Folder](https://drive.google.com/drive/folders/1QLEnT5B0p43axdNkIvrdnVWpWZgaTeaT)

I'm providing you with eviction warrant data for the whole state of Maryland from 2022 through December 2024. These are from the same District Court of Maryland and Department of Housing and Community Development (DHCD) [source](https://app.powerbigov.us/view?r=eyJrIjoiYWI1Yzg0YjYtNDFkZS00MDUyLThlMDctYmE1ZjY5MGI0MWJhIiwidCI6IjdkM2I4ZDAwLWY5YmUtNDZlNy05NDYwLTRlZjJkOGY3MzE0OSJ9&pageName=ReportSection) as the data from Exercise 2, but also include street addresses. While these data are technically public, it is best practice not to store address-level data on a public GitHub repository. It is also a best practice not to commit large raw data files to Git. For both these reasons, I have shared this dataset in a [Google Drive](https://drive.google.com/drive/folders/1QLEnT5B0p43axdNkIvrdnVWpWZgaTeaT) folder to which your UMD account has been invited. You should download `md_eviction_warrants_through_2024.csv` store it in the exercise03 directory on your computer before starting to code. 

There is a `.gitignore` file in the exercise03 directory that prevents any `.csv` file from being tracked by Git. As long as you don't modify this `.gitignore`, the raw data file won't get committed, pushed to your remote fork, or included in a pull request back to the course repo.

## File Management and Submitting
To submit, please:
1. Make a new branch on your fork for this exercise.
2. Make a notebook for your exercise with your first name as an underscored suffix (e.g., `exercise02_chester.ipynb`)
    - You can either copy this notebook to work off of or start with a fresh notebook. Your choice.
4. Make commits to that branch as you work on the exercise.
5. Don't commit the eviction warrant CSV or other raw data files to Git.
    - Instead, please add any other raw data files your analysis depends on the [Exercise 3 Google Drive Folder](https://drive.google.com/drive/folders/1QLEnT5B0p43axdNkIvrdnVWpWZgaTeaT).
    - The current `.gitignore` will prevent CSV files from committing. Add additional file names/extensions as necessary.
6. Make a pull request from your branch. Ensure that the only files included in your pull request are those you intended for this exercise.

## Getting Started
To get started, here's some code I developed for geocoding the address in each eviction warrant into a geographic coordinate. You can include all or parts of this code in your own exercise, or just run this notebook to produce the `md_eviction_warrants_through_2024.geoparquet` file and import it into your own notebook to use the results.

### U.S. Census Geocoder
This geocoding process makes use of a [free geocoder provided by the US Census](https://geocoding.geo.census.gov/geocoder/). It's not the most accurate geocoder available, but it's free and fast.

### Breaking Code Into Modules
In this geocoding process, I'm demonstrating an approach to coding where you break code up into multiple modules and then import names between modules. This helps keep things tidy, allows you to easily reuse code that's generalizable between applications (e.g., the `utils.py` module here), and organize code used for more specific purposes (e.g., the `exercise03.py` and `census_geocode.py` modules).

This is exactly how packages work——modules are the basic building blocks. If you wrote an interconnected set of modules to address a certain problem space, you could publish it as a package and let others download it with conda or pip. That's how open-source software gets its start!






In [2]:
import pandas as pd
import geopandas as gpd
import utils
import exercise03
import census_geocode

%load_ext autoreload
%autoreload 2

The research question answered by the project below is: what is the rate of warrants, evictions, and rate of eviction from warrants in 2024 per the number of rental households by county within the state of Maryland. The dataset brought in to corroborate the eviction data provided for the assignment was from the National Historic Geographic Information Systems (NHGIS). The specific dataset was the 2019-2024 ACS Five-Year estimates tenure data, which shows the number of renter and owner households. Dwelling ownership rate can vary drastically between geographies, so adjusting the rate of warrants and evictions by rental households provides a more accurate picture to what places are experiencing a disproportionate rate of eviction. The outline for the process started with cleaning the eviction dataset. After cleaning the data, there were two datasets created with the eviction database, one for only evictions and one for all warrants. Both of these datasets were summarized by county. Then, after reformatting Baltimore city in both data sets, the few important columns were subsetted from all datasets. All three datasets were joined together and were used to create the eviction and warrant rates per 1,000 renter households within the counties. The most noticeable result of this process was Baltimore County had the highest rates of warrants and evictions within the State of Maryland. Harford County had a high warrant rate, but a less severe rate of evictions. Interestingly, the City of Baltimore appeared to have the highest rate of success for evictions of any county in Maryland, but despite this its eviction rate for all renter households was just under half of Baltimore County’s rate.

In [3]:
# Load warrants and make sure zip codes are stored as strings without decimals
warrants_df = pd.read_csv('md_eviction_warrants_through_sept2025.csv')

# Ensure zip codes are stored as strings
warrants_df['TenantZipCode'] = warrants_df['TenantZipCode'].astype('Int64').astype('string')
warrants_df['EventDate'] = pd.to_datetime(warrants_df['EventDate'])
warrants_df['EvictedDate'] = pd.to_datetime(warrants_df['EvictedDate'])
warrants_df['SourceDate'] = pd.to_datetime(warrants_df['SourceDate'])

len(warrants_df) # How many warrants are we working with?

607881

In [4]:
# Rather than geocoding 600K+ addresses, can we get only the unique ones?
geocode_input_df = exercise03.prep_warrants_for_geocoding(warrants_df)

607881 warrants input
Reduced to 205620 unique addresses


In [5]:
# The Census Geocoder API can only accept up to 10K rows at a time, so we have to break
# our dataframe into chunks

# Split into dataframes with less than 10K rows each
geocode_input_dfs = utils.chunk_dataframe(geocode_input_df, 9999)

# Save each dataframe as a CSV without a header
utils.save_dfs_to_csv(geocode_input_dfs, 'geocode_inputs', header=False)

split dataframe into 21 chunks


In [6]:
# Geocode addresses with the Census Geocoder (set test=True to process only one file)
census_geocode.geocode_csvs('geocode_inputs', 'geocode_outputs', test=True)

TEST MODE: Processing only one file.
Saved results to: geocode_outputs/geocoderesult_df_14.csv


In [7]:
# Recombine outputs from geocoder into a single dataframe
geocode_output_df = exercise03.combine_census_geocoded_csvs('geocode_outputs')
len(geocode_output_df)

9999

In [8]:
# Merge geocoded address back onto the inputs with separate fields for address, city, state, and zip
geocoded_df = geocode_input_df.merge(geocode_output_df, left_index=True, right_index=True)
len(geocoded_df)

9999

In [9]:
# Use address, city, state, and zip columns to join geocodes onto original warrant records
warrants_df = warrants_df.merge(geocoded_df, on=['TenantAddress','TenantCity','TenantState','TenantZipCode'])
len(warrants_df)

50754

In [10]:
# Convert warrants into a geodataframe with points
warrants_gdf = utils.lonlat_str_to_geodataframe(warrants_df, 'match_lon_lat')
len(warrants_gdf)

50754

In [11]:
# What proportion of records have points?
len(warrants_gdf[warrants_gdf.lon.notnull()]) / len(warrants_gdf)

0.9530086298616858

In [12]:
# What proportion of records have exact geocode matches?
len(warrants_gdf[warrants_gdf.match_type == 'Exact']) / len(warrants_gdf)

0.45738266934625843

In [13]:
#Cleaning up years
warrants_df['Year'] = warrants_df['Year'].astype('Int64').astype('string')
warrants_df['EvictionYear'] = warrants_df['EvictionYear'].astype('Int64').astype('string')

In [14]:
#Looking at types of warrants and events
warrants_df['EventType'].unique()

array(['Warrant of Restitution - Return of Service - Cancelled',
       'Warrant of Restitution - Return of Service - Evicted',
       'Petition - For Warrant of Restitution Filed',
       'Warrant of Restitution - Return of Service - Expired'],
      dtype=object)

In [15]:
#Filtering for only cases where the tenant was evicted
warrants_evicted = warrants_df[warrants_df['EventType'] == 'Warrant of Restitution - Return of Service - Evicted']
#Filtering for tenants evicted in 2024
warrants_evicted_2024 = warrants_evicted[warrants_evicted['Year'] == '2024']
len(warrants_evicted_2024)

258

In [16]:
#Using groupby to create a dataframe summarized by county
evictions_county = warrants_evicted_2024.groupby('County')['EventType'].size().reset_index(name="evictions")

In [17]:
#Importing data
Tenure = pd.read_csv('nhgis0152_ds272_20245_county.csv')

In [18]:
#Replace county in the first column
Tenure['COUNTY'] = Tenure['COUNTY'].replace(' County', '', regex = True)
#Replace City with city in original data
evictions_county['County'] = evictions_county['County'].replace(' City', ' city', regex = True)

In [19]:
#Joining Data
eviction_data = evictions_county.merge(Tenure, left_on='County', right_on='COUNTY')

In [42]:
#Creating a new column based on evictions by rental housing units
eviction_data['evictions_per_1k_renter_hslds'] = (eviction_data['evictions'] / eviction_data['AUUEE003']) * 1000

In [41]:
#Repeating Process above but for warrants
#Filtering for only cases where the tenant was evicted
events = ['Warrant of Restitution - Return of Service - Evicted',
'Warrant of Restitution - Return of Service - Cancelled',
'Warrant of Restitution - Return of Service - Expired']
warrants_all = warrants_df[warrants_df['EventType'].isin(events)]
#Filtering for tenants evicted in 2024
warrants_all_2024 = warrants_all[warrants_all['Year'] == '2024']
#Using groupby to create a dataframe
warrants_county = warrants_all_2024.groupby('County')['EventType'].size().reset_index(name="warrants")
#Importing data
tenure_warrants = pd.read_csv('nhgis0152_ds272_20245_county.csv')
#Replace county in the first column
tenure_warrants['COUNTY'] = tenure_warrants['COUNTY'].replace(' County', '', regex = True)
#Replace City with city in original data
warrants_county['County'] = warrants_county['County'].replace(' City', ' city', regex = True)
#Joining Data
warrants_data = warrants_county.merge(tenure_warrants, left_on='County', right_on='COUNTY')
#Creating a new column based on evictions by rental housing units
warrants_data['warrants_per_1k_renter_hslds'] = (warrants_data['warrants'] / warrants_data['AUUEE003']) * 1000

In [46]:
#Cleaning what I need from the warrants data
clean_warrants_df = warrants_data[['County',
                                   'warrants_per_1k_renter_hslds',
                                   'warrants']]
#Cleaning what I need from the evictions data
clean_eviction_df = eviction_data[['County',
                                   'evictions_per_1k_renter_hslds',
                                   'evictions']]

In [47]:
#Cleaning and joining the warrants and evictions tables
final_data = clean_warrants_df.merge(clean_eviction_df, left_on='County', right_on='County')
final_data['eviction_warrant_ratio'] = final_data['evictions'] / final_data['warrants']

In [48]:
print(final_data)

             County  warrants_per_1k_renter_hslds  warrants  \
0      Anne Arundel                      2.272077       127   
1         Baltimore                     14.638985      1638   
2    Baltimore city                      0.625945        84   
3             Cecil                      9.563658        96   
4           Charles                      2.290006        25   
5         Frederick                      2.329451        56   
6           Harford                     10.578177       219   
7            Howard                      3.301286       114   
8        Montgomery                      0.755158       102   
9   Prince George's                      1.342978       176   
10       Washington                      2.917212        58   
11         Wicomico                      2.947534        45   

    evictions_per_1k_renter_hslds  evictions  eviction_warrant_ratio  
0                        0.196794         11                0.086614  
1                        1.108202     

['Unnamed: 0',
 'ID',
 'EventDate',
 'EventType',
 'EventComment',
 'County',
 'Location',
 'TenantAddress',
 'TenantCity',
 'TenantState',
 'TenantZipCode',
 'CaseType',
 'CaseNumber',
 'EvictedDate',
 'Source',
 'SourceDate',
 'Year',
 'EvictionYear',
 'unique_id',
 'input_address',
 'match_status',
 'match_type',
 'match_address',
 'match_lon_lat',
 'match_tiger_line_id',
 'match_tiger_line_side',
 'lon',
 'lat',
 'geometry']

In [61]:
#Bonus Question, creating a new column to determine accuracy
warrants_gdf_short = warrants_gdf[['EventType',
                                   'match_status',
                                   'match_type',
                                   'CaseType']]
warrants_gdf_short['match_cmb'] = warrants_gdf_short['match_status'] + '_' + warrants_gdf_short['match_type']

/var/folders/nr/xvc_gl8x14vg9gfzn6mr60fw0000gn/T/ipykernel_5589/3925499318.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warrants_gdf_short['match_cmb'] = warrants_gdf_short['match_status'] + '_' + warrants_gdf_short['match_type']


In [62]:
#Replacing na values
warrants_gdf_short['match_cmb'] = warrants_gdf_short['match_cmb'].fillna('No_Match')

/var/folders/nr/xvc_gl8x14vg9gfzn6mr60fw0000gn/T/ipykernel_5589/3290015455.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warrants_gdf_short['match_cmb'] = warrants_gdf_short['match_cmb'].fillna('No_Match')


In [75]:
#Creating a pivot counting the intersections of the two column values
frequency_event_types = pd.crosstab(warrants_gdf_short['EventType'],warrants_gdf_short['match_cmb'])
frequency_event_types

match_cmb,Match_Exact,Match_Non_Exact,No_Match
EventType,,,
Petition - For Warrant of Restitution Filed,15143,16369,1531
Warrant of Restitution - Return of Service - Cancelled,6483,6862,693
Warrant of Restitution - Return of Service - Evicted,802,965,78
Warrant of Restitution - Return of Service - Expired,786,959,83


In [76]:
#Frequency of Match_exact for different warrant types
frequency_event_types['Match_Exact_Rate'] = (frequency_event_types['Match_Exact'] / (frequency_event_types['Match_Exact'] +
                                                                                    frequency_event_types['Match_Non_Exact'] +
                                                                                    frequency_event_types['No_Match'])) * 100
#Frequency of Match_Non_Exact for different warrant types
frequency_event_types['Match_Non_Exact_Rate'] = (frequency_event_types['Match_Non_Exact'] / (frequency_event_types['Match_Exact'] +
                                                                                    frequency_event_types['Match_Non_Exact'] +
                                                                                    frequency_event_types['No_Match'])) * 100
#Frequency of No_Match for different warrant types
frequency_event_types['No_Match_Rate'] = (frequency_event_types['No_Match'] / (frequency_event_types['Match_Exact'] +
                                                                                    frequency_event_types['Match_Non_Exact'] +
                                                                                    frequency_event_types['No_Match'])) * 100

In [77]:
frequency_event_types

match_cmb,Match_Exact,Match_Non_Exact,No_Match,Match_Exact_Rate,Match_Non_Exact_Rate,No_Match_Rate
EventType,,,,,,
Petition - For Warrant of Restitution Filed,15143,16369,1531,45.828163,49.538480,4.633357
Warrant of Restitution - Return of Service - Cancelled,6483,6862,693,46.181792,48.881607,4.936601
Warrant of Restitution - Return of Service - Evicted,802,965,78,43.468835,52.303523,4.227642
Warrant of Restitution - Return of Service - Expired,786,959,83,42.997812,52.461707,4.540481


In [72]:
#Creating a pivot counting the intersections of the two column values
frequency_case_types = pd.crosstab(warrants_gdf_short['CaseType'],warrants_gdf_short['match_cmb'])
frequency_case_types

match_cmb,Match_Exact,Match_Non_Exact,No_Match
CaseType,,,
Breach of Lease,40,74,4
Failure to Pay Rent,22933,24794,2362
Tenant Holding Over,122,130,9
Wrongful Detainer,119,157,10


In [73]:
#Frequency of Match_exact for different case types
frequency_case_types['Match_Exact_Rate'] = (frequency_case_types['Match_Exact'] / (frequency_case_types['Match_Exact'] +
                                                                                    frequency_case_types['Match_Non_Exact'] +
                                                                                    frequency_case_types['No_Match'])) * 100
#Frequency of Match_Non_Exact for different warrant types
frequency_case_types['Match_Non_Exact_Rate'] = (frequency_case_types['Match_Non_Exact'] / (frequency_case_types['Match_Exact'] +
                                                                                    frequency_case_types['Match_Non_Exact'] +
                                                                                    frequency_case_types['No_Match'])) * 100
#Frequency of No_Match for different warrant types
frequency_case_types['No_Match_Rate'] = (frequency_case_types['No_Match'] / (frequency_case_types['Match_Exact'] +
                                                                                    frequency_case_types['Match_Non_Exact'] +
                                                                                    frequency_case_types['No_Match'])) * 100

In [74]:
frequency_case_types

match_cmb,Match_Exact,Match_Non_Exact,No_Match,Match_Exact_Rate,Match_Non_Exact_Rate,No_Match_Rate
CaseType,,,,,,
Breach of Lease,40,74,4,33.898305,62.711864,3.389831
Failure to Pay Rent,22933,24794,2362,45.784504,49.499890,4.715606
Tenant Holding Over,122,130,9,46.743295,49.808429,3.448276
Wrongful Detainer,119,157,10,41.608392,54.895105,3.496503


For the bonus question, I did two types of analysis based on the case type and the event type. I created a new column which was a join of the text from the match type and match columns from the evictions data. I also used data from all years available within the dataset. I found for the types of warrants, petitions and cancelled evictions had marginally higher no matches and exact matches than the evictions and expired warrants. On the other hand, evictions and expired warrants had higher non-exact match rates. As for the distribution of match rates, there was a much more drastic difference between Breach of Lease warrants and petitions than the other three types of cases. 62.7 percent of breach of lease cases were non-exact matches. The failure to pay rent and tenant holding over cases both had between 49 and 50 percent of their geocoded points as non-exact matches, but failure to pay rent had a slightly lower rate of exact matches at 45.8% and a slightly higher rate of no matches at 4.7 percent. Wrongful detainer cases had a slightly higher rate non-exact matches at around 55 percent, alongside a lower exact match rate of 41 percent and a no match rate of 3.5 percent. 